In [1]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import os

# === RUTA DE ENTRADA ===
gbff_file = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\Zaire_ebolavirus_genome\ncbi_dataset\data\GCA_000848505.1\genomic.gbff"

# === RUTA DE SALIDA ===
output_dir = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\Blancos_terapeutico"
os.makedirs(output_dir, exist_ok=True)

# === ARCHIVO RESUMEN ===
resumen_tsv = os.path.join(output_dir, "resumen_proteinas.tsv")
with open(resumen_tsv, "w", encoding="utf-8") as resumen:
    resumen.write("Locus_Tag\tProducto\tInicio\tFin\tStrand\tLongitud_AA\tArchivo_Fasta\n")

    for record in SeqIO.parse(gbff_file, "genbank"):
        for feature in record.features:
            if feature.type == "CDS" and "translation" in feature.qualifiers:
                # === Datos ===
                locus_tag = feature.qualifiers.get("locus_tag", ["NA"])[0]
                product = feature.qualifiers.get("product", ["NA"])[0]
                start = int(feature.location.start) + 1
                end = int(feature.location.end)
                strand = feature.location.strand
                seq_aa = feature.qualifiers["translation"][0]

                # === Crear registro FASTA ===
                protein_id = f"{locus_tag}_{product.replace(' ', '_')[:20]}"
                seq_record = SeqRecord(Seq(seq_aa), id=protein_id, description=f"{product} [{locus_tag}]")
                fasta_filename = os.path.join(output_dir, f"{protein_id}.fasta")
                
                # === Guardar archivo FASTA individual ===
                with open(fasta_filename, "w") as fasta_out:
                    SeqIO.write(seq_record, fasta_out, "fasta")
                
                # === Guardar en resumen ===
                resumen.write(f"{locus_tag}\t{product}\t{start}\t{end}\t{strand}\t{len(seq_aa)}\t{protein_id}.fasta\n")

print(f"Extracción completada. Archivos guardados en: {output_dir}")


Extracción completada. Archivos guardados en: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\Blancos_terapeutico
